<a href="https://colab.research.google.com/github/wantuds-crypto/Ciencia-datos-curso/blob/main/Final_precios_inmuebles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Introducción y objetivo del análisis
### El problema
**Problemática:** predecir o explicar el precio de venta de una vivienda a partir de sus características físicas y de ubicación, para apoyar decisiones como fijar el precio de listado, detectar propiedades sub o sobrevaloradas y entender qué atributos elevan más el valor de una casa.

El precio de una vivienda depende de muchas cosas a la vez: su tamaño, su estado, su antigüedad, si tiene vista o frente al agua y, sobre todo, **dónde está**. Para quien compra, vende, construye o financia vivienda, entender **qué características se asocian con un precio más alto** permite tomar mejores decisiones: fijar precios de lista, priorizar zonas de inversión, decidir si conviene renovar o detectar ventas con precios anómalos.

### Objetivo general
Identificar, con los datos disponibles, qué características de la vivienda y de su ubicación se relacionan más con su **precio de venta**, y dejar el terreno preparado (datos limpios, variables construidas y evaluadas) para un futuro modelo de estimación de precios.

### Preguntas que queremos responder
1. ¿Qué tan confiables son los datos y qué problemas de calidad tienen?
2. ¿Cómo se distribuye el precio de venta? ¿Hay valores imposibles o sospechosos?
3. ¿Qué zonas y ciudades concentran las ventas y cuáles tienen precios más altos?
4. ¿Cómo cambia el precio según el tamaño, la antigüedad, el estado y la calidad de la vista?
5. ¿Qué variables (originales o construidas) tienen la relación más fuerte con el precio?
6. ¿Qué recomendaciones prácticas y qué próximos pasos se derivan?
\

In [131]:
# --- Librerías ---
from pathlib import Path
import re
import warnings
import os
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import duckdb
from scipy import stats
import kagglehub

In [132]:
# Descarga del dataset
path = kagglehub.dataset_download("debayank2024/house-price-prediction")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'house-price-prediction' dataset.
Path to dataset files: /kaggle/input/house-price-prediction


In [133]:
df = pd.read_csv(os.path.join(path, "modified_data.csv"))
print("Filas y columnas:", df.shape)
df.info()
df.head()

Filas y columnas: (4600, 18)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            4600 non-null   object 
 1   price           4600 non-null   float64
 2   bedrooms        4600 non-null   float64
 3   bathrooms       4600 non-null   float64
 4   sqft_living     4600 non-null   int64  
 5   sqft_lot        4600 non-null   int64  
 6   floors          4600 non-null   float64
 7   waterfront      4600 non-null   int64  
 8   view            4600 non-null   int64  
 9   condition       4600 non-null   int64  
 10  sqft_above      4600 non-null   int64  
 11  sqft_basement   4600 non-null   int64  
 12  yr_built        4600 non-null   int64  
 13  yr_renovated    4600 non-null   int64  
 14  street          4600 non-null   object 
 15  city            4600 non-null   object 
 16  statezip        4600 non-null   object 
 17  pric

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,price_per_sqft
0,2014-05-02,"313,000.00",3.00,1.50,1340,7912,1.50,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133,233.58
1,2014-05-02,"2,384,000.00",5.00,2.50,3650,9050,2.00,0,4,5,3370,280,1921,0,709 W Blaine St,Seattle,WA 98119,653.15
2,2014-05-02,"342,000.00",3.00,2.00,1930,11947,1.00,0,0,4,1930,0,1966,0,26206-26214 143rd Ave SE,Kent,WA 98042,177.20
3,2014-05-02,"420,000.00",3.00,2.25,2000,8030,1.00,0,0,4,1000,1000,1963,0,857 170th Pl NE,Bellevue,WA 98008,210.00
4,2014-05-02,"550,000.00",4.00,2.50,1940,10500,1.00,0,0,4,1140,800,1976,1992,9105 170th Ave NE,Redmond,WA 98052,283.51


In [134]:
df.describe()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,price_per_sqft
count,"4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00"
mean,"551,962.99",3.40,2.16,"2,139.35","14,852.52",1.51,0.01,0.24,3.45,"1,827.27",312.08,"1,970.79",808.61,265.88
std,"563,834.70",0.91,0.78,963.21,"35,884.44",0.54,0.08,0.78,0.68,862.17,464.14,29.73,979.41,357.50
min,0.00,0.00,0.00,370.00,638.00,1.00,0.00,0.00,1.00,370.00,0.00,"1,900.00",0.00,0.00
25%,"322,875.00",3.00,1.75,"1,460.00","5,000.75",1.00,0.00,0.00,3.00,"1,190.00",0.00,"1,951.00",0.00,180.82
50%,"460,943.46",3.00,2.25,"1,980.00","7,683.00",1.50,0.00,0.00,3.00,"1,590.00",0.00,"1,976.00",0.00,243.86
75%,"654,962.50",4.00,2.50,"2,620.00","11,001.25",2.00,0.00,0.00,4.00,"2,300.00",610.00,"1,997.00","1,999.00",314.84
max,"26,590,000.00",9.00,8.00,"13,540.00","1,074,218.00",3.50,1.00,4.00,5.00,"9,410.00","4,820.00","2,014.00","2,014.00","22,533.90"


## 1. Formato Fechas
-pasar el date a formato fecha y además agregar una columna solo con el mes para determinar si hay meses que las viviendas son más costosas que otros.

In [135]:
df["date"] = pd.to_datetime(df["date"])
df = df.assign(
    mes=df['date'].dt.month
)

In [136]:
df2 = df.drop(columns=["street"])
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   date            4600 non-null   datetime64[ns]
 1   price           4600 non-null   float64       
 2   bedrooms        4600 non-null   float64       
 3   bathrooms       4600 non-null   float64       
 4   sqft_living     4600 non-null   int64         
 5   sqft_lot        4600 non-null   int64         
 6   floors          4600 non-null   float64       
 7   waterfront      4600 non-null   int64         
 8   view            4600 non-null   int64         
 9   condition       4600 non-null   int64         
 10  sqft_above      4600 non-null   int64         
 11  sqft_basement   4600 non-null   int64         
 12  yr_built        4600 non-null   int64         
 13  yr_renovated    4600 non-null   int64         
 14  city            4600 non-null   object        
 15  stat

In [137]:
df.select_dtypes(include=['bool']).info()
df.select_dtypes(include=['object']).info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Empty DataFrame
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   street    4600 non-null   object
 1   city      4600 non-null   object
 2   statezip  4600 non-null   object
dtypes: object(3)
memory usage: 107.9+ KB


| No. | Problemática | Descripción | Columna(s) afectada(s) |
|---|---|---|---|
| 1 | Ubicación fragmentada | La ubicación está repartida en `street`, `city` y `statezip`, con demasiadas categorías únicas para usarlas directamente en un modelo; hay que decidir cómo resumirla (por ciudad, código postal o zona). | `street`, `city`, `statezip` |
| 2 | Variables redundantes / fuga de información | `sqft_above` + `sqft_basement` ≈ `sqft_living`, y `price_per_sqft` ya se calculó a partir de `price` y `sqft_living`; puede usarse para predecir `price` | `sqft_above`, `sqft_basement`, `sqft_living`, `price_per_sqft` |
| 3 | Valor centinela mal interpretado | `yr_renovated` = 0 no significa "año cero", sino "nunca renovacion"; tratarlo como año real distorsiona promedios y modelos. | `yr_renovated` |
| 4 | Escalas ordinales tratadas como numéricas | `condition` y `view` son categorías ordenadas, no cantidades continuas; hay que decidir si tratarlas como categóricas ordenadas o asumir distancia uniforme entre niveles. | `condition`, `view` |
| 5 | Desbalance de clases | `waterfront` probablemente tiene muy pocos casos en 1, lo que puede sesgar cualquier análisis o modelo que la use como variable importante. | `waterfront` |
| 6 | Estacionalidad / tendencia temporal | `date` permite ver si los precios varían por época del año, pero exige tratarla como serie de tiempo y no solo como etiqueta. | `date` |



## 2. Limpieza de datos

- Eliminamos duplicados exactos.
- Imputamos nulos (mediana para numéricas, moda para categóricas).
- Casteamos cada columna al tipo que le corresponde.
- Corregimos el valor centinela `yr_renovated == 0` (no es "año cero", es "nunca renovada").
- Hay Viviendas Precios =0.
-se eliminan estas columnas porque street cuenta con muchos datos unicos

### Preguntas y decisiones para el trabajo

- por que razon dejamos los datos nulos
- por que razon usamos la media
- por que razon eliminados 100 registros y asi

In [138]:
print("Nulos por columna:")
print(df.isnull().sum())

print("\nFilas duplicadas exactas:", df.duplicated().sum())

Nulos por columna:
date              0
price             0
bedrooms          0
bathrooms         0
sqft_living       0
sqft_lot          0
floors            0
waterfront        0
view              0
condition         0
sqft_above        0
sqft_basement     0
yr_built          0
yr_renovated      0
street            0
city              0
statezip          0
price_per_sqft    0
mes               0
dtype: int64

Filas duplicadas exactas: 0


In [139]:
dup_exactos = df.duplicated().sum()
dup_sin_fecha = df.drop(columns="date").duplicated().sum()
clave_dir = ["street", "city", "statezip"]
filas_dir_repetida = df.duplicated(clave_dir, keep=False).sum()
grupos_dir = df[df.duplicated(clave_dir, keep=False)].groupby(clave_dir, observed=False)
atributos_fisicos = ["bedrooms", "bathrooms", "sqft_living", "sqft_lot", "yr_built"]
grupos_distintos = (grupos_dir[atributos_fisicos].nunique() > 1).any(axis=1).sum()

print(f"Filas duplicadas exactas: {dup_exactos}")
print(f"Filas duplicadas ignorando la fecha (posible doble registro): {dup_sin_fecha}")
print(f"Filas que comparten dirección con otra fila: {filas_dir_repetida} "
      f"({grupos_dir.ngroups} direcciones), de las cuales {grupos_distintos} direcciones tienen atributos físicos distintos")

nulos = pd.DataFrame({
    "nulos": df.isna().sum(),
    "% nulos": (df.isna().mean() * 100).round(2),
    "ceros": (df == 0).sum()
})
# "Nulos ocultos": ceros que en realidad significan 'dato no disponible'
nulos["nulos ocultos (0 sin sentido)"] = 0
for c in ["price", "bedrooms", "bathrooms"]:
    nulos.loc[c, "nulos ocultos (0 sin sentido)"] = (df[c] == 0).sum()
nulos.T

Filas duplicadas exactas: 0
Filas duplicadas ignorando la fecha (posible doble registro): 0
Filas que comparten dirección con otra fila: 145 (71 direcciones), de las cuales 69 direcciones tienen atributos físicos distintos


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,price_per_sqft,mes
nulos,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
% nulos,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
ceros,0.00,49.00,2.00,2.00,0.00,0.00,0.00,"4,567.00","4,140.00",0.00,0.00,"2,745.00",0.00,"2,735.00",0.00,0.00,0.00,49.00,0.00
nulos ocultos (0 sin sentido),0.00,49.00,2.00,2.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


**Analisis de nulos y duplicados.** No hay filas duplicadas ni celdas vacías.

-hay ceros que no tienen sentido, como precios iguales a 0 y viviendas con 0 habitaciones y 0 baños.

-Estos son *nulos ocultos* y para tartarlos en la limpieza.

-Las direcciones repetidas corresponden casi siempre a viviendas diferentes (distinta área, habitaciones o año).

-Tenemos conjuntos de casas o apartamentos con la misma dirección base, por lo cual no son duplicados.

In [140]:
# Casteo de tipos
df['date'] = pd.to_datetime(df['date'])
df['waterfront'] = df['waterfront'].astype(bool)

for c in ['city', 'street', 'statezip']:
    df[c] = df[c].astype('category')

cat_condition = pd.CategoricalDtype(categories=sorted(df['condition'].unique()), ordered=True)
df['condition'] = df['condition'].astype(cat_condition)

cat_view = pd.CategoricalDtype(categories=sorted(df['view'].unique()), ordered=True)
df['view'] = df['view'].astype(cat_view)

df.dtypes

,0
date,datetime64[ns]
price,float64
bedrooms,float64
bathrooms,float64
sqft_living,int64
sqft_lot,int64
floors,float64
waterfront,bool
view,category
condition,category


In [141]:
# Booleana: 2 estados que representan sí/no
df['waterfront'] = df['waterfront'].astype(bool)

# Nominal: categorías sin orden -> category simple
df['city'] = df['city'].astype('category')

# Ordinal: categorías CON orden -> category con ordered=True
cat_condition = pd.CategoricalDtype(categories=[1,2,3,4,5], ordered=True)
df['condition'] = df['condition'].astype(cat_condition)

###Analsis Inicial:
- Nominales: `street`, `city`, `statezip`
- Ordinales: `condition` , `view` , `floors`
- Booleanas: `waterfront`
- Numéricas(no nominal / ordinal / booleana):`price`, `sqft_living`, `sqft_lot`, `sqft_above`, `sqft_basement`, `price_per_sqft`,`bedrooms`, `bathrooms`, `yr_built`, `yr_renovated`,`yr_renovated`, `date`


In [142]:
# Filas donde price o sqft_living son 0 o negativos
filas_invalidas = df[(df['price'] <= 0) | (df['sqft_living'] <= 0)]
print("Filas inválidas:", len(filas_invalidas))
df = df[(df['price'] > 0) & (df['sqft_living'] > 0)].reset_index(drop=True)

Filas inválidas: 49


In [148]:
# Agrupación geográfica de ciudades en zonas:
ZONAS = {
    "Seattle": ["Seattle"],
    "Eastside": ["Bellevue", "Redmond", "Kirkland", "Sammamish", "Issaquah", "Mercer Island", "Medina",
                 "Clyde Hill", "Yarrow Point", "Newcastle", "Woodinville", "Bothell", "Kenmore",
                 "Beaux Arts Village", "Inglewood-Finn Hill"],
    "Norte": ["Shoreline", "Lake Forest Park"],
    "Sur": ["Renton", "Kent", "Auburn", "Federal Way", "Burien", "Des Moines", "Tukwila", "SeaTac",
            "Normandy Park", "Covington", "Maple Valley", "Algona", "Pacific", "Milton", "Black Diamond", "Enumclaw"],
    "Rural / Cascadas": ["Snoqualmie", "North Bend", "Duvall", "Carnation", "Fall City", "Preston",
                         "Ravensdale", "Skykomish", "Snoqualmie Pass", "Vashon"],
}
CIUDAD_A_ZONA = {c: z for z, ciudades in ZONAS.items() for c in ciudades}


## Creamos llaves para poder hacer un análisis relacional real con JOINs sin inventar informacion.

| Tabla del modelo | | Llave primaria |,
|---|---|---|---|
| `inmuebles` | | `id_inmueble`
| `ventas` | | `id_venta` |
| `dim_ciudades` | | `ciudad` |
| `referencia_zip` | | `codigo_postal`|
              

## Caracteristicas de Variables para tener en cuenta en los features

| Columna | Tipo de dato | Descripción | Rol en el análisis | Observaciones de calidad |
|---|---|---|---|---|
| `date` | `datetime64` | Fecha de la venta | Fecha | Viene como texto; debe convertirse a fecha |
| `price` | `float (USD)` | Precio de venta en dólares | Variable objetivo (*target*) | Hay precios en 0 y valores extremos que no tienen sentido |
| `bedrooms` | `entero` | Número de habitaciones | Variable numérica | Viene como decimal; los valores 0 son poco creíbles |
| `bathrooms` | `decimal` | Número de baños; por ejemplo, 0.25 equivale a medio baño sin ducha | Variable numérica | Los valores 0 son poco creíbles |
| `sqft_living` | `entero` | Área habitable construida, en pies² | Variable numérica | Puede tener asimetría y valores muy altos |
| `sqft_lot` | `entero` | Área total del lote o terreno, en pies² | Variable numérica | Es muy asimétrica por lotes rurales o terrenos demasiado grandes |
| `floors` | `decimal` | Número de pisos; 1.5 representa un piso y medio | Variable numérica | Sin observaciones críticas iniciales |
| `waterfront` | `booleano` | Indica si la vivienda tiene frente al agua: 1 = sí, 0 = no | Variable categórica binaria | Variable muy desbalanceada: menos del 1 % tiene valor 1 |
| `view` | `entero ordinal (0–4)` | Calificación de la vista: 0 = sin vista destacada y 4 = excelente | Variable categórica ordinal | La mayoría de las viviendas tiene calificación 0 |
| `condition` | `entero ordinal (1–5)` | Estado de conservación: 1 = malo y 5 = excelente | Variable categórica ordinal | Hay pocas viviendas con calificaciones entre 1 y 2 |
| `sqft_above` | `entero` | Área construida sobre el nivel del suelo, en pies² | Variable numérica | Debe ser consistente con `sqft_living` y `sqft_basement` |
| `sqft_basement` | `entero` | Área de sótano, en pies²; 0 significa que no tiene sótano | Variable numérica | El 0 representa “no tiene sótano”, no un dato faltante |
| `yr_built` | `entero` | Año de construcción de la vivienda | Variable temporal | No debe ser mayor que el año de venta |
| `yr_renovated` | `entero / nulo` | Año de la última renovación; 0 significa que nunca se ha renovado | Variable temporal | El 0 es un valor centinela; no puede ser anterior al año de construcción |
| `street` | `texto` | Dirección de la vivienda: calle y número | Variable de texto | Puede repetirse, tener espacios innecesarios y no funciona como llave única |
| `city` | `categoría` | Ciudad donde se ubica la vivienda | Posible llave para JOIN | Puede tener diferencias de mayúsculas, espacios o categorías con pocos registros |
| `statezip` | `texto → estado + código postal` | Estado y código postal juntos; ejemplo: `WA 98133` |

## Paso 3 · Feature engineering: variables temporales.

In [145]:
df['ratio_bano_habitacion'] = df['bathrooms'] / df['bedrooms'].replace(0, np.nan)
df['ratio_bano_habitacion'] = df['ratio_bano_habitacion'].fillna(0)

df['tiene_sotano'] = df['sqft_basement'] > 0
df['ratio_living_lot'] = df['sqft_living'] / df['sqft_lot']
df['ratio_above_living'] = df['sqft_above'] / df['sqft_living']

df['precio_prom_ciudad'] = df.groupby('city', observed=True)['price'].transform('mean')
df['habitaciones_totales'] = df['bedrooms'] + df['bathrooms']

df['log_price'] = np.log1p(df['price'])
df['log_sqft_living'] = np.log1p(df['sqft_living'])

df.filter(regex='ratio_|sotano|prom_ciudad|log_|habitaciones_totales').head()

,ratio_bano_habitacion,tiene_sotano,ratio_living_lot,ratio_above_living,precio_prom_ciudad,habitaciones_totales,log_price,log_sqft_living
0,0.50,False,0.17,1.00,"420,392.36",4.50,12.65,7.20
1,0.50,True,0.40,0.92,"584,294.90",7.50,14.68,8.20
2,0.67,False,0.16,1.00,"441,880.99",5.00,12.74,7.57
3,0.75,True,0.25,0.50,"862,255.05",5.25,12.95,7.60
4,0.62,True,0.18,0.59,"667,649.53",6.50,13.22,7.57


In [146]:
#Variables categóricas
texto_cols = ["street", "city", "statezip"]
diag_texto = pd.DataFrame({
    "valores únicos": raw[texto_cols].nunique(),
    "con espacios al inicio/fin": [(raw[c] != raw[c].str.strip()).sum() for c in texto_cols],
    "con espacios dobles": [raw[c].str.contains(r"\s{2,}", regex=True).sum() for c in texto_cols],
    "únicos tras normalizar (minúsculas+trim)": [raw[c].str.strip().str.lower().nunique() for c in texto_cols],
})
diag_texto["categorías duplicadas por formato"] = diag_texto["valores únicos"] - diag_texto["únicos tras normalizar (minúsculas+trim)"]
display(diag_texto)

patron_statezip = r"^[A-Z]{2} \d{5}$"
print("statezip con formato distinto de 'XX 99999':", (~raw["statezip"].str.match(patron_statezip)).sum())
print("Estados distintos:", raw["statezip"].str[:2].unique().tolist())
zip_multi_ciudad = raw.assign(zip=raw["statezip"].str[-5:]).groupby("zip")["city"].nunique()
print(f"Códigos postales asociados a más de una ciudad: {(zip_multi_ciudad > 1).sum()} de {zip_multi_ciudad.size}")

frec_ciudad = raw["city"].value_counts()
print(f"\nCiudades: {frec_ciudad.size}. Ciudades con menos de {MIN_VENTAS_CIUDAD} ventas: {(frec_ciudad < MIN_VENTAS_CIUDAD).sum()}")
frec_ciudad.to_frame("ventas").T

NameError: name 'raw' is not defined

## 4. EDA: Revisamos distribución del precio, outliers, relaciones con variables numéricas y con ubicación.